# Phase 8: Cannibalization & Promotional Impact Analysis

Cross-family correlation analysis and promotional lift quantification

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from retail_iq.config import RAW_DATA_DIR, PLOT_DIR
from retail_iq.preprocessing import load_raw_data, preprocess_dates, merge_datasets

## 1. Load Data

In [ ]:
train, test, stores, oil, holidays, transactions = load_raw_data()
train, test, oil, holidays, transactions = preprocess_dates([train, test, oil, holidays, transactions])
df = merge_datasets(train, stores, oil, holidays, transactions)

print(f'Loaded: {len(df)} rows')
print(f'Stores: {df["store_nbr"].nunique()}, Families: {df["family"].nunique()}')

## 2. Cannibalization Detection

In [ ]:
def compute_residual_sales(df):
    """Detrend sales using 28-day rolling mean."""
    df = df.copy()
    df['sales_trend'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(28, min_periods=1).mean()
    )
    df['sales_resid'] = df['sales'] - df['sales_trend']
    return df

def find_cannibal_pairs(df, promo_threshold=0, corr_threshold=-0.35):
    """Find negatively correlated family pairs during promotion periods."""
    df = compute_residual_sales(df)
    stores = df['store_nbr'].unique()
    pairs = []
    
    for s in stores:
        df_s = df[df['store_nbr'] == s]
        promo_mask = df_s['onpromotion'] > promo_threshold
        
        if promo_mask.sum() < 10:
            continue
            
        df_promo = df_s[promo_mask].pivot(index='date', columns='family', values='sales_resid')
        corr = df_promo.corr()
        
        families = corr.columns
        for i, fam_i in enumerate(families):
            for fam_j in families[i+1:]:
                r = corr.loc[fam_i, fam_j]
                if pd.isna(r):
                    continue
                # P-value
                n = promo_mask.sum()
                t_stat = r * np.sqrt((n-2)/(1-r**2))
                p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n-2))
                
                if r < corr_threshold and p_value < 0.05:
                    pairs.append({
                        'store': s,
                        'family_i': fam_i,
                        'family_j': fam_j,
                        'r': r,
                        'p_value': p_value,
                        'n_obs': n
                    })
    
    return pd.DataFrame(pairs)

cannibal_pairs = find_cannibal_pairs(df)
print(f'Found {len(cannibal_pairs)} candidate pairs')
print(cannibal_pairs.head(10))

In [ ]:
# Top cannibalization pairs
if len(cannibal_pairs) > 0:
    top_pairs = cannibal_pairs.nsmallest(min(5, len(cannibal_pairs)), 'r')
    print('\nTop 5 Cannibalization Pairs (most negative correlation):')
    print(top_pairs[['family_i', 'family_j', 'r', 'p_value']].to_string(index=False))
else:
    print('No significant cannibalization pairs found with r < -0.35')

In [ ]:
# Cross-family correlation heatmap (sample store)
sample_store = df['store_nbr'].unique()[0]
df_s = df[df['store_nbr'] == sample_store]
df_pivot = df_s.pivot(index='date', columns='family', values='sales')

plt.figure(figsize=(16, 12))
sns.heatmap(df_pivot.corr(), cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title(f'Cross-Family Sales Correlation - Store {sample_store}')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/cross_family_correlation_store{sample_store}.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Promotional Lift Analysis

In [ ]:
def compute_promo_lift(df, window_pre=28):
    """
    Compute promotional lift: (actual - counterfactual) / counterfactual
    Counterfactual = 4-week rolling pre-promotion mean
    """
    df = df.sort_values(['store_nbr', 'family', 'date']).copy()
    
    # Baseline: rolling mean of previous 28 days
    df['baseline'] = df.groupby(['store_nbr', 'family'])['sales'].transform(
        lambda x: x.shift(1).rolling(window_pre, min_periods=1).mean()
    )
    
    # Lift calculation
    df['lift'] = (df['sales'] - df['baseline']) / df['baseline']
    df['lift'] = df['lift'].replace([np.inf, -np.inf], np.nan)
    
    # Filter to promotion events
    promo_events = df[df['onpromotion'] > 0].copy()
    
    return promo_events

promo_lift = compute_promo_lift(df)
print(f'Promotional events: {len(promo_lift)}')
print(f'\nLift statistics:')
print(promo_lift['lift'].describe())

In [ ]:
# Promotional lift by family
lift_by_family = promo_lift.groupby('family')['lift'].agg(['mean', 'std', 'count']).reset_index()
lift_by_family = lift_by_family[lift_by_family['count'] >= 10]
lift_by_family = lift_by_family.sort_values('mean', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(lift_by_family['family'][:10], lift_by_family['mean'][:10])
plt.xlabel('Mean Promotional Lift')
plt.title('Top 10 Families by Promotional Lift')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/promo_lift_by_family.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: onpromotion count vs lift
plt.figure(figsize=(10, 6))
plt.scatter(promo_lift['onpromotion'], promo_lift['lift'], alpha=0.3)
plt.xlabel('Items on Promotion')
plt.ylabel('Promotional Lift')
plt.title('Promotion Depth vs Lift')
plt.axhline(y=0, color='r', linestyle='--')
plt.ylim(-2, 5)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/promo_depth_vs_lift.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Save Cannibalization Report

In [ ]:
# Generate report
report_lines = [
    '# Cannibalization & Promotional Impact Analysis Report\n',
    '\n',
    '## Summary\n',
    f'- Total cannibalization pairs identified: {len(cannibal_pairs)}\n',
    f'- Promotional events analyzed: {len(promo_lift)}\n',
    '\n',
    '## Cannibalization Pairs (r < -0.35, p < 0.05)\n',
]

if len(cannibal_pairs) > 0:
    report_lines.append(cannibal_pairs.to_markdown(index=False))
else:
    report_lines.append('No significant pairs found.\n')

report_lines.extend([
    '\n',
    '## Promotional Lift by Family (Top 10)\n',
    lift_by_family.head(10).to_markdown(index=False),
    '\n',
    '## Methodology\n',
    '- Cannibalization: Pearson correlation on residual sales during promotion periods\n',
    '- Residual sales = actual - 28-day rolling mean (detrended)\n',
    '- Lift = (actual - counterfactual) / counterfactual, counterfactual = 4-week pre-promo mean\n',
    '\n',
    '## Visualizations\n',
    '- cross_family_correlation_store{N}.png\n',
    '- promo_lift_by_family.png\n',
    '- promo_depth_vs_lift.png\n',
])

# Save report
from pathlib import Path
report_path = Path('outputs/reports/cannibalization_report.md')
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, 'w') as f:
    f.write('\n'.join(report_lines))

print(f'Report saved to: {report_path}')

## Summary

- Cannibalization pairs identified using r < -0.35 threshold on residual sales
- Promotional lift quantified for all promotion events
- Report generated with findings and methodology

Deliverables complete for academic submission.